In [1]:
import importlib.util
import sys


def _missing(mod):
    return importlib.util.find_spec(mod) is None


to_install = []
if _missing("tabpfn"):
    to_install.append("tabpfn")
if _missing("tabpfn_client"):
    to_install.append("tabpfn-client")

if to_install:
    print("Installing:", to_install)
    get_ipython().run_line_magic(
        "pip", "install -q --no-warn-conflicts --break-system-packages " + " ".join(to_install)
    )
    print("Done. If imports fail, restart the kernel and re-run.")
else:
    print("tabpfn / tabpfn-client already available.")

Installing: ['tabpfn', 'tabpfn-client']
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 753.2/753.2 kB 13.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 33.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 370.4/370.4 kB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.8 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.
Done. If imports fail, restart the kernel and re-run.


In [3]:
import tabpfn_client
from tabpfn_client import TabPFNClassifier, TabPFNRegressor, set_access_token
from sklearn.model_selection import train_test_split
import pandas as pd

# Authenticate
API_TOKEN = "tabpfn_sk_AsJ9rJKua_1gkC0Ffj8eiaPfE10tW78vFIbbxpk_6Ew"
tabpfn_client.set_access_token(API_TOKEN)

# Load data
df = pd.read_csv("/kaggle/input/datasets/amirmahdidaraei/vlst-data/VLST.csv")
leakage_cols = ["NO.", "Name", "Time since stent implantation"]
df = df.drop(leakage_cols, axis=1)
target_col = "Stent thrombosis"
X, y = df.drop(target_col, axis=1), df[target_col]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Train model
model = TabPFNClassifier()
model.fit(X_train, y_train)

# Predict
predictions = model.predict(X_test)
print("Predictions:", predictions)

00:11 Fitting... Done!
00:05 Predicting... Done!
Predictions: [0 0 0 ... 0 0 0]


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    f1_score,
    fbeta_score,
    precision_score,
    recall_score,
    RocCurveDisplay,
    PrecisionRecallDisplay,
    ConfusionMatrixDisplay,
)

# None -> tune on OOF train scores; float -> fixed threshold
DECISION_THRESHOLD = None
THRESHOLD_STRATEGY = "f1"
THRESHOLD_BETA = 2.0
THRESHOLD_CV_SPLITS = 10
RANDOM_STATE = 42


def pos_proba(clf, X):
    classes = list(clf.classes_)
    idx = classes.index(1) if 1 in classes else 1
    return np.asarray(clf.predict_proba(X)[:, idx], dtype=float)


def oof_pos_proba(X, y, n_splits=10, seed=42):
    oof = np.zeros(len(y), dtype=float)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    for k, (tr, va) in enumerate(skf.split(X, y), 1):
        m = TabPFNClassifier()
        m.fit(X.iloc[tr], y.iloc[tr])
        oof[va] = pos_proba(m, X.iloc[va])
        print(f"  OOF fold {k}/{n_splits}")
    return oof


def select_threshold(y_true, scores, *, strategy="f2", beta=2.0, grid_points=199):
    grid = np.linspace(0.01, 0.99, grid_points)
    P, R, F1, FB = [], [], [], []
    for t in grid:
        pred = (scores >= t).astype(int)
        P.append(precision_score(y_true, pred, zero_division=0))
        R.append(recall_score(y_true, pred, zero_division=0))
        F1.append(f1_score(y_true, pred, zero_division=0))
        FB.append(fbeta_score(y_true, pred, beta=beta, zero_division=0))
    P, R, F1, FB = map(np.asarray, (P, R, F1, FB))

    if strategy == "f1":
        i = int(np.argmax(F1))
    elif strategy in ("f2", "fbeta"):
        i = int(np.argmax(FB))
    else:
        raise ValueError(f"Unknown strategy: {strategy}")

    info = {
        "precision": float(P[i]),
        "recall": float(R[i]),
        "f1": float(F1[i]),
        f"f{beta:g}": float(FB[i]),
    }
    return float(grid[i]), info


if DECISION_THRESHOLD is None:
    print(f"Tuning threshold via {THRESHOLD_CV_SPLITS}-fold OOF on X_train (strategy={THRESHOLD_STRATEGY})...")
    oof_scores = oof_pos_proba(X_train, y_train, n_splits=THRESHOLD_CV_SPLITS, seed=RANDOM_STATE)
    THRESHOLD, thr_info = select_threshold(
        y_train, oof_scores, strategy=THRESHOLD_STRATEGY, beta=THRESHOLD_BETA
    )
    print(f"Chosen threshold={THRESHOLD:.4f} | OOF " + " ".join(f"{k}={v:.3f}" for k, v in thr_info.items()))
else:
    THRESHOLD = float(DECISION_THRESHOLD)
    print(f"Using fixed DECISION_THRESHOLD={THRESHOLD}")

y_prob = pos_proba(model, X_test)
y_pred = (y_prob >= THRESHOLD).astype(int)

print(classification_report(y_test, y_pred, zero_division=0))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

metrics = {
    "accuracy": float((y_pred == y_test).mean()),
    "precision": float(precision_score(y_test, y_pred, zero_division=0)),
    "recall": float(recall_score(y_test, y_pred, zero_division=0)),
    "f1": float(f1_score(y_test, y_pred, zero_division=0)),
    "roc_auc": float(roc_auc_score(y_test, y_prob)),
    "pr_auc": float(average_precision_score(y_test, y_prob)),
    "decision_threshold": float(THRESHOLD),
}
for k, v in metrics.items():
    print(f"{k:>18}: {v:.4f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
RocCurveDisplay.from_predictions(y_test, y_prob, ax=axes[0], name="TabPFN")
axes[0].set_title(f"ROC (AUC={metrics['roc_auc']:.3f})")
PrecisionRecallDisplay.from_predictions(y_test, y_prob, ax=axes[1], name="TabPFN")
axes[1].set_title(f"Precision-Recall (AP={metrics['pr_auc']:.3f})")
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    labels=[0, 1],
    display_labels=["neg (0)", "pos (1)"],
    ax=axes[2],
    colorbar=False,
    values_format="d",
)
axes[2].set_title(f"Confusion @ t={THRESHOLD:.3f}")
plt.tight_layout()
plt.show()

metrics